# Import Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("zefang-liu/phishing-email-dataset")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/616 [00:00<?, ?B/s]

Phishing_Email.csv:   0%|          | 0.00/52.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18650 [00:00<?, ? examples/s]

In [ ]:
import pandas as pd

df = pd.DataFrame(dataset['train'])
df.head()

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [ ]:
df = df[["Email Text", "Email Type"]]
df.columns = ["text", "label"]
df.head()

,text,label
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,the other side of * galicismos * * galicismo *...,Safe Email
2,re : equistar deal tickets are you still avail...,Safe Email
3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [ ]:
df["label_encoded"] = df["label"].apply(lambda x: 1 if x == "Phishing Email" else 0)
df.head()

,text,label,label_encoded
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email,0
1,the other side of * galicismos * * galicismo *...,Safe Email,0
2,re : equistar deal tickets are you still avail...,Safe Email,0
3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email,1
4,software at incredibly low prices ( 86 % lower...,Phishing Email,1


In [ ]:
df = df.drop_duplicates()
df = df.dropna()
print(len(df))

17538


# Near-Duplicate Removal

After doing the basic cleaning, we're removing texts that are exact same or similar by comparing the lengths and jaccard similarity. This allows removing texts that are similar in template/pattern.

In [ ]:
import pandas as pd
import re
from collections import defaultdict

def clean_email_text(text):
    """Basic text cleaning for duplicate detection."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)   # remove URLs
    text = re.sub(r"[^a-z0-9\s]", " ", text)      # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()      # normalize spaces
    return text

def get_word_ngrams(text, n=3):
    words = text.split()
    if len(words) < n:
        return {tuple(words)} if words else set()
    return set(tuple(words[i:i+n]) for i in range(len(words)-n+1))

def jaccard_similarity(set1, set2):
    if not set1 and not set2:
        return 1.0
    union = set1 | set2
    if not union:
        return 0.0
    return len(set1 & set2) / len(union)

def make_block_key(cleaned_text, length_bucket_size=25, top_words=5):
    words = cleaned_text.split()
    unique_words = sorted(set(words))[:top_words]
    length_bucket = len(words) // length_bucket_size
    return (length_bucket, tuple(unique_words))

def deduplicate_by_ngrams(
    df,
    text_col="text",
    n=3,
    sim_threshold=0.8,
    length_bucket_size=25,
    top_words=5,
    return_dropped=False
):

    working = df.copy().reset_index(drop=False).rename(columns={"index": "original_index"})

    # Step 1: clean text
    working["cleaned_text"] = working[text_col].apply(clean_email_text)

    # Step 2: remove exact duplicates on cleaned text
    exact_dup_mask = working.duplicated(subset=["cleaned_text"], keep="first")
    exact_dropped = working[exact_dup_mask].copy()
    working = working[~exact_dup_mask].copy().reset_index(drop=True)

    # Step 3: precompute n-gram sets
    working["ngrams"] = working["cleaned_text"].apply(lambda x: get_word_ngrams(x, n=n))

    # Step 4: create blocks
    working["block_key"] = working["cleaned_text"].apply(
        lambda x: make_block_key(x, length_bucket_size=length_bucket_size, top_words=top_words)
    )

    # Step 5: compare only within blocks
    to_drop = set()
    blocks = defaultdict(list)

    for idx, row in working.iterrows():
        blocks[row["block_key"]].append(idx)

    for block_indices in blocks.values():
        if len(block_indices) < 2:
            continue

        kept = []
        for idx in block_indices:
            if idx in to_drop:
                continue

            current_ngrams = working.at[idx, "ngrams"]
            is_duplicate = False

            for kept_idx in kept:
                sim = jaccard_similarity(current_ngrams, working.at[kept_idx, "ngrams"])
                if sim >= sim_threshold:
                    to_drop.add(idx)
                    is_duplicate = True
                    break

            if not is_duplicate:
                kept.append(idx)

    near_dup_dropped = working.loc[list(to_drop)].copy()
    deduped = working.drop(index=list(to_drop)).copy()

    # cleanup columns
    dropped_df = pd.concat([exact_dropped, near_dup_dropped], ignore_index=True)
    deduped = deduped.drop(columns=["ngrams", "block_key"])
    dropped_df = dropped_df.drop(columns=["ngrams", "block_key"], errors="ignore")

    # sort back by original order if desired
    deduped = deduped.sort_values("original_index").reset_index(drop=True)
    dropped_df = dropped_df.sort_values("original_index").reset_index(drop=True)

    if return_dropped:
        return deduped, dropped_df

    return deduped

In [ ]:
deduped_df, dropped_df = deduplicate_by_ngrams(
    df,
    text_col="text",
    n=2,
    sim_threshold=0.6,
    return_dropped=True
)

print("Original rows:", len(df))
print("After deduplication:", len(deduped_df))
print("Dropped rows:", len(dropped_df))

Original rows: 17538
After deduplication: 16597
Dropped rows: 941


In [ ]:
deduped_df["text_length"] = deduped_df["cleaned_text"].str.split().str.len()

deduped_df.groupby("label_encoded")["text_length"].describe()

,count,mean,std,min,25%,50%,75%,max
label_encoded,,,,,,,,
0,10893.0,535.580373,23946.218341,1.0,73.0,158.0,316.00,2498497.0
1,5704.0,266.396038,508.510057,0.0,57.0,117.0,262.25,11744.0


In [ ]:
max_length = 300  # or 10000 depending on tolerance

deduped_df = deduped_df[
    deduped_df["text_length"] <= max_length
]
deduped_df = deduped_df[
    deduped_df["text_length"] > 20
]

In [ ]:
deduped_df.groupby("label_encoded")["text_length"].describe()

,count,mean,std,min,25%,50%,75%,max
label_encoded,,,,,,,,
0,7515.0,129.454558,76.205087,21.0,64.0,117.0,187.0,300.0
1,4096.0,110.427979,68.048946,21.0,55.0,91.0,152.0,300.0


In [ ]:
deduped_df.head()

,original_index,text,label,label_encoded,cleaned_text,text_length
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email,0,re 6 1100 disc uniformitarianism re 1086 sex l...,180
1,1,the other side of * galicismos * * galicismo *...,Safe Email,0,the other side of galicismos galicismo is a sp...,73
2,2,re : equistar deal tickets are you still avail...,Safe Email,0,re equistar deal tickets are you still availab...,209
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email,1,hello i am your hot lil horny toy i am the one...,107
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email,1,software at incredibly low prices 86 lower dra...,64


In [ ]:
def further_clean(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"\S+@\S+", " <EMAIL> ", text)
    text = re.sub(r"\b\d+\b", " <NUM> ", text)
    text = re.sub(r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\b", " <DATE> ", text)
    text = re.sub(r"\b(account|invoice|order|id|number)\s*\w*", r"\1 <ID>", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

deduped_df["cleaned_text"] = deduped_df["cleaned_text"].apply(further_clean)

In [ ]:
deduped_df["label"].value_counts()

,count
label,
Safe Email,7515
Phishing Email,4096


In [ ]:
from sklearn.model_selection import train_test_split

x = deduped_df["cleaned_text"]
y = deduped_df["label_encoded"]

# split into train(80) and temp (20)
x_train, x_temp, y_train, y_temp = train_test_split(
    x, y,
    random_state=3,
    test_size=0.2,
    stratify=y)

print(x_train.shape)
print(x_temp.shape)

(9288,)
(2323,)


In [ ]:
x_valid, x_test, y_valid, y_test = train_test_split(
    x_temp, y_temp,
    random_state=3,
    test_size=0.5,
    stratify=y_temp)

print(x_valid.shape)
print(x_test.shape)

(1161,)
(1162,)


# Test with TF-IDF + LR
We're testing out with our traditional ML model if the previous cleaning/deduplication stpes have made any difference.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# capturing unigrams & bigrams
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))

x_train_tfidf = tfidf.fit_transform(x_train)
x_valid_tfidf = tfidf.transform(x_valid)

In [ ]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(max_iter=1000, random_state=42)
classifier.fit(x_train_tfidf, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
# predict on validation set
y_pred = classifier.predict(x_valid_tfidf)
y_prob = classifier.predict_proba(x_valid_tfidf)[:, 1]

# evaluate
from sklearn.metrics import classification_report, accuracy_score

print("Accuracy: ", accuracy_score(y_valid, y_pred))
print(classification_report(y_valid, y_pred))

#print(f"\nConfusion Matrix: \n {confusion_matrix(y_valid, y_pred)}")


Accuracy:  0.9732988802756245
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       752
           1       0.99      0.93      0.96       409

    accuracy                           0.97      1161
   macro avg       0.98      0.96      0.97      1161
weighted avg       0.97      0.97      0.97      1161



In [ ]:
x_test_tfidf = tfidf.transform(x_test)

y_pred = classifier.predict(x_test_tfidf)
y_prob = classifier.predict_proba(x_test_tfidf)[:, 1]

# evaluate
print("Accuracy: ", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy:  0.96815834767642
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       752
           1       0.97      0.94      0.95       410

    accuracy                           0.97      1162
   macro avg       0.97      0.96      0.96      1162
weighted avg       0.97      0.97      0.97      1162



Despite the deduplication process, the model still performs very well in distinguishing phishing and safe emails.

This demonstrates that there are no data leakage across sets, meaning there are no exact same emails or similar emails that may let model to learn beforehand.

However, this doesn't imply that the model has successfully learned the semantic differences between phishing and safe emails. This has been verified in our main notebook where we dive into EDA and explore different areas that may affect the classification process, including domain and phrase overlappings.

In [ ]:
from collections import Counter

phish_words = Counter(" ".join(deduped_df[deduped_df.label_encoded==1]["cleaned_text"]).split())
safe_words = Counter(" ".join(deduped_df[deduped_df.label_encoded==0]["cleaned_text"]).split())

In [ ]:
print(f"Phishing Words: {phish_words}")
print(f"Safe Words: {safe_words}")

Phishing Words: Counter({'<NUM>': 24833, 'to': 10396, 'the': 10119, 'you': 8846, 'and': 7679, 'a': 6723, 'your': 5818, 'of': 5330, 'for': 4929, 'is': 4100, 'in': 3856, 'this': 3589, 'we': 2899, 'our': 2644, 'it': 2390, 'i': 2387, 'from': 2286, 's': 2256, 'have': 2255, 'are': 2253, 'here': 2237, 'with': 2229, 'on': 2228, 'be': 2148, 'com': 1993, 'or': 1892, 'that': 1836, 'no': 1805, '<ID>': 1796, 'at': 1698, 'http': 1666, 'not': 1663, 'all': 1636, 'if': 1621, 'will': 1610, 'more': 1576, 'click': 1528, 'free': 1490, 'as': 1478, 'email': 1470, 'can': 1443, 'please': 1366, 'e': 1340, 'by': 1292, 'get': 1284, 'now': 1225, 't': 1203, 'do': 1171, 'out': 1081, 'us': 944, 'time': 941, 'an': 917, 'only': 883, 'new': 876, 'www': 828, 'one': 814, 'just': 802, 'up': 786, 'mail': 764, 'r': 751, 'o': 751, 'information': 743, 'online': 737, 'message': 734, 'order': 728, 'list': 727, 'my': 724, 'best': 718, '<DATE>': 716, 'm': 694, 'like': 693, 'net': 689, 'site': 686, 'any': 660, 'over': 644, 'want': 

In [ ]:
len(deduped_df)

11611

In [ ]:
print(phish_words.most_common(20))
print(safe_words.most_common(20))

[('<NUM>', 24833), ('to', 10396), ('the', 10119), ('you', 8846), ('and', 7679), ('a', 6723), ('your', 5818), ('of', 5330), ('for', 4929), ('is', 4100), ('in', 3856), ('this', 3589), ('we', 2899), ('our', 2644), ('it', 2390), ('i', 2387), ('from', 2286), ('s', 2256), ('have', 2255), ('are', 2253)]
[('<NUM>', 65681), ('the', 34120), ('to', 24109), ('and', 16732), ('of', 16566), ('a', 15621), ('i', 13674), ('in', 11765), ('for', 11089), ('is', 9925), ('on', 9075), ('you', 8870), ('that', 7892), ('it', 7328), ('this', 6937), ('be', 6040), ('s', 5715), ('with', 5282), ('have', 5223), ('from', 4726)]


No noticeable differences in most common words -- mostly consisted of stopwords or generic words.

The mere frequency count isn't effective in filtering out keywords.